# 01 - Cổng lượng tử 1 qubit

Cổng lượng tử = **ma trận unitary** $U$ (thỏa $U^\dagger U = I$). Áp cổng lên trạng thái
chính là nhân ma trận:

$$|\psi'\rangle = U|\psi\rangle$$

Unitary kéo theo hai hệ quả quan trọng:

- **Bảo toàn chuẩn** - tổng xác suất vẫn bằng 1.
- **Luôn đảo ngược được** - mọi cổng đều có cổng nghịch đảo $U^\dagger$. Không có cổng
  lượng tử nào "xóa" thông tin (khác hẳn cổng AND cổ điển).

Trên Bloch sphere, mọi cổng 1 qubit là một **phép quay** quanh một trục nào đó.

In [ ]:
import numpy as np

from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector
from qiskit.visualization import plot_bloch_multivector, plot_histogram
from qiskit_aer.primitives import SamplerV2
from IPython.display import display

import qiskit
import qiskit_aer

print("qiskit:", qiskit.__version__, "| qiskit-aer:", qiskit_aer.__version__)


def ket(sv, nd=3):
    """In statevector dưới dạng tổng các ket, bỏ qua amplitude ~ 0."""
    phan = []
    for i, bien_do in enumerate(sv.data):
        if abs(bien_do) < 1e-9:
            continue
        bits = format(i, f"0{sv.num_qubits}b")
        if abs(bien_do.imag) < 1e-9:
            he_so = f"{bien_do.real:+.{nd}f}"
        elif abs(bien_do.real) < 1e-9:
            he_so = f"{bien_do.imag:+.{nd}f}i"
        else:
            he_so = f"({bien_do.real:+.{nd}f}{bien_do.imag:+.{nd}f}i)"
        phan.append(f"{he_so}|{bits}>")
    return "  ".join(phan) if phan else "0"


def probs(sv, nd=3):
    """Xác suất đo được từng kết quả, key là chuỗi bit thường (không phải np.str_)."""
    return {str(k): round(float(v), nd) for k, v in sv.probabilities_dict().items()}


def mat(obj, nd=3):
    """Ma trận của một cổng hoặc cả mạch, đã làm tròn."""
    return np.round(Operator(obj).data, nd)


r2 = 1 / np.sqrt(2)
zero = Statevector.from_label("0")
one = Statevector.from_label("1")
plus = Statevector([r2, r2])

## 1. Nhóm Pauli: X, Y, Z

$$X=\begin{pmatrix}0&1\\1&0\end{pmatrix}\quad
  Y=\begin{pmatrix}0&-i\\i&0\end{pmatrix}\quad
  Z=\begin{pmatrix}1&0\\0&-1\end{pmatrix}$$

- **X** - "NOT lượng tử", bit flip $|0\rangle \leftrightarrow |1\rangle$. Quay $180^\circ$ quanh trục X.
- **Z** - phase flip: $|1\rangle \to -|1\rangle$, còn $|0\rangle$ giữ nguyên. Quay $180^\circ$ quanh trục Z.
- **Y** - vừa bit flip vừa phase flip ($Y = iXZ$). Quay $180^\circ$ quanh trục Y.

Cả ba đều thỏa $X^2 = Y^2 = Z^2 = I$: làm hai lần thì trở về chỗ cũ.

In [ ]:
for nhan in ["I", "X", "Y", "Z"]:
    print(f"{nhan} =")
    print(np.round(Operator.from_label(nhan).data, 3), "\n")

print("Tác dụng lên các ket cơ sở:")
for nhan in ["X", "Y", "Z"]:
    op = Operator.from_label(nhan)
    for ten, s in [("|0>", zero), ("|1>", one)]:
        print(f"  {nhan}{ten} = {ket(s.evolve(op))}")

## 2. Hadamard: cổng tạo superposition

$$H = \frac{1}{\sqrt2}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$$

$H$ chuyển qua lại giữa cơ sở Z và cơ sở X:

$$H|0\rangle = |+\rangle,\quad H|1\rangle = |-\rangle,\quad H|+\rangle = |0\rangle,\quad H^2 = I$$

Vì thế "thêm $H$ trước khi đo" chính là **đo trong cơ sở X**. Một đẳng thức rất hay dùng:
$HZH = X$ (đổi cơ sở thì phase flip biến thành bit flip).

In [ ]:
X = Operator.from_label("X")
Z = Operator.from_label("Z")
H = Operator.from_label("H")
S = Operator.from_label("S")
T = Operator.from_label("T")
I = Operator.from_label("I")

print("H =\n", np.round(H.data, 3))
print("\nH|0> =", ket(zero.evolve(H)))
print("H|1> =", ket(one.evolve(H)))
print("\nH @ H = I ?", np.allclose(H.data @ H.data, I.data))
print("H Z H = X ?", np.allclose(H.data @ Z.data @ H.data, X.data))

## 3. Cổng phase: S, T, P($\lambda$)

$$S=\begin{pmatrix}1&0\\0&i\end{pmatrix}\quad
  T=\begin{pmatrix}1&0\\0&e^{i\pi/4}\end{pmatrix}\quad
  P(\lambda)=\begin{pmatrix}1&0\\0&e^{i\lambda}\end{pmatrix}$$

Chúng chỉ thêm phase cho thành phần $|1\rangle$, tức **quay quanh trục Z** trên Bloch sphere:
$S$ quay $90^\circ$, $T$ quay $45^\circ$, $Z$ quay $180^\circ$. Quan hệ:

$$S^2 = Z, \qquad T^2 = S, \qquad P(\pi/2) = S$$

Vì chúng không đổi $|\alpha|, |\beta|$ nên **đo ngay sau cổng phase thì không thấy gì khác**.
Phase chỉ lộ ra khi kết hợp với $H$ (interference) - đây là cơ chế cốt lõi của mọi thuật toán lượng tử.

In [ ]:
print("S =\n", np.round(S.data, 3))
print("T =\n", np.round(T.data, 3))

print("\nS^2 = Z ?", np.allclose(S.data @ S.data, Z.data))
print("T^2 = S ?", np.allclose(T.data @ T.data, S.data))

qc_p = QuantumCircuit(1)
qc_p.p(np.pi / 2, 0)
print("P(pi/2) = S ?", np.allclose(mat(qc_p), np.round(S.data, 3)))

# S không đổi xác suất, nhưng đổi trạng thái
print("\nS|+> =", ket(plus.evolve(S)), "-> xác suất", probs(plus.evolve(S)))
print("|+>  =", ket(plus), "-> xác suất", probs(plus))
print("Chỉ lộ ra sau khi cho qua H:")
print("  H|+>  =", ket(plus.evolve(H)))
print("  H S|+> =", ket(plus.evolve(S).evolve(H)))

## 4. Cổng quay theo góc tùy ý: RX, RY, RZ

Pauli cho góc $180^\circ$ cố định. Muốn góc bất kỳ thì dùng cổng rotation:

$$R_X(\theta)=\begin{pmatrix}\cos\frac\theta2 & -i\sin\frac\theta2\\
                             -i\sin\frac\theta2 & \cos\frac\theta2\end{pmatrix}\qquad
  R_Y(\theta)=\begin{pmatrix}\cos\frac\theta2 & -\sin\frac\theta2\\
                             \sin\frac\theta2 & \cos\frac\theta2\end{pmatrix}\qquad
  R_Z(\theta)=\begin{pmatrix}e^{-i\theta/2}&0\\0&e^{i\theta/2}\end{pmatrix}$$

Chú ý $\theta/2$: phải quay $\theta = 2\pi$ mới về đúng trạng thái ban đầu (kể cả phase).

$R_Y$ đặc biệt tiện khi học vì nó giữ amplitude **thực**, nên chỉnh được xác suất một cách trực quan:
$P(1) = \sin^2(\theta/2)$.

In [ ]:
import matplotlib.pyplot as plt

print(f"{'theta':>8} {'P(0)':>8} {'P(1)':>8}   (RY)")
for th in np.linspace(0, np.pi, 5):
    qc = QuantumCircuit(1)
    qc.ry(th, 0)
    p = Statevector(qc).probabilities()
    print(f"{th:8.4f} {p[0]:8.3f} {p[1]:8.3f}")

In [ ]:
# Vẽ đường cong P(1) theo theta
goc = np.linspace(0, 2 * np.pi, 200)
p1 = []
for th in goc:
    qc = QuantumCircuit(1)
    qc.ry(th, 0)
    p1.append(Statevector(qc).probabilities()[1])

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(goc, p1, label="P(1) do mô phỏng")
ax.plot(goc, np.sin(goc / 2) ** 2, "--", label=r"$\sin^2(\theta/2)$")
ax.set_xlabel("theta (rad)")
ax.set_ylabel("P(1)")
ax.set_title("RY(theta) tác dụng lên |0>")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Quay RY trên Bloch sphere: |0> đi từ cực bắc về phía +X rồi xuống cực nam
for th in [0, np.pi / 4, np.pi / 2, np.pi]:
    qc = QuantumCircuit(1)
    qc.ry(th, 0)
    print(f"RY({th:.4f})")
    display(plot_bloch_multivector(Statevector(qc)))

## 5. Global phase: `==` khác `equiv`

$R_X(\pi) = -iX$ - khác $X$ đúng một hệ số $-i$. Hệ số này là **global phase**, không quan sát
được, nên về mặt vật lý hai cổng làm đúng một việc.

- `Operator(a) == Operator(b)` - so sánh **từng phần tử ma trận** (nghiêm ngặt).
- `Operator(a).equiv(Operator(b))` - so sánh **bỏ qua global phase** (thường là cái bạn cần).

In [ ]:
qc_rx = QuantumCircuit(1)
qc_rx.rx(np.pi, 0)

print("RX(pi) =\n", mat(qc_rx))
print("X      =\n", np.round(X.data, 3))
print("\nbằng nhau tuyệt đối ?", Operator(qc_rx) == X)
print("tương đương (bỏ global phase) ?", Operator(qc_rx).equiv(X))

## 6. Thứ tự: mạch đọc trái sang phải, ma trận nhân phải sang trái

Đây là chỗ rất dễ nhầm. Mạch vẽ theo thứ tự thời gian:

```
     ┌───┐┌───┐
q_0: ┤ H ├┤ X ├     H trước, X sau
     └───┘└───┘
```

nhưng ma trận tương ứng là:

$$U = X \cdot H$$

Cổng **áp sau** nằm **bên trái** trong tích ma trận, vì $X(H|\psi\rangle) = (XH)|\psi\rangle$.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)  # áp trước
qc.x(0)  # áp sau
print(qc.draw())

print("\nOperator(qc) =\n", mat(qc))
print("\nX @ H        =\n", np.round(X.data @ H.data, 3))
print("\nH @ X        =\n", np.round(H.data @ X.data, 3))
print("\nOperator(qc) == X @ H ?", np.allclose(Operator(qc).data, X.data @ H.data))

## 7. Nghịch đảo và cổng tổng quát U

Vì mọi cổng đều unitary nên `qc.inverse()` luôn tồn tại: nó đảo ngược thứ tự các cổng và
lấy liên hợp Hermite từng cổng ($T^\dagger$ viết là `tdg`, $S^\dagger$ là `sdg`).

Cổng tổng quát nhất của 1 qubit là $U(\theta, \varphi, \lambda)$ - mọi cổng 1 qubit đều là
một trường hợp riêng của nó:

$$U(\theta,\varphi,\lambda)=\begin{pmatrix}
\cos\frac\theta2 & -e^{i\lambda}\sin\frac\theta2\\
e^{i\varphi}\sin\frac\theta2 & e^{i(\varphi+\lambda)}\cos\frac\theta2\end{pmatrix}$$

In [ ]:
qc = QuantumCircuit(1)
qc.t(0)
qc.h(0)
qc.s(0)

print("mạch gốc:")
print(qc.draw())
print("\nnghịch đảo:")
print(qc.inverse().draw())

tong = qc.compose(qc.inverse())
print("\nmạch + nghịch đảo = I ?", Operator(tong).equiv(I))

qc_u = QuantumCircuit(1)
qc_u.u(np.pi / 2, 0, np.pi, 0)
print("\nU(pi/2, 0, pi) = H ?", np.allclose(Operator(qc_u).data, H.data))

## Tóm tắt các cổng 1 qubit

Ký hiệu $\mathrm{diag}(a, b)$ là ma trận đường chéo $\begin{pmatrix}a&0\\0&b\end{pmatrix}$.

| Cổng | Qiskit | Ma trận | Quay trên Bloch | Ghi nhớ |
|---|---|---|---|---|
| $X$ | `qc.x(0)` | $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ | $180^\circ$ quanh $X$ | NOT lượng tử |
| $Y$ | `qc.y(0)` | $\begin{pmatrix}0&-i\\i&0\end{pmatrix}$ | $180^\circ$ quanh $Y$ | bit flip + phase flip |
| $Z$ | `qc.z(0)` | $\mathrm{diag}(1, -1)$ | $180^\circ$ quanh $Z$ | phase flip |
| $H$ | `qc.h(0)` | $\frac{1}{\sqrt2}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ | $180^\circ$ quanh trục $\frac{X+Z}{\sqrt2}$ | tạo superposition |
| $S$ | `qc.s(0)` | $\mathrm{diag}(1, i)$ | $90^\circ$ quanh $Z$ | $S^2 = Z$ |
| $T$ | `qc.t(0)` | $\mathrm{diag}(1, e^{i\pi/4})$ | $45^\circ$ quanh $Z$ | $T^2 = S$ |
| $P(\lambda)$ | `qc.p(lam, 0)` | $\mathrm{diag}(1, e^{i\lambda})$ | $\lambda$ quanh $Z$ | tổng quát của $Z, S, T$ |
| $R_X(\theta)$ | `qc.rx(th, 0)` | $\cos\frac{\theta}{2} I - i\sin\frac{\theta}{2} X$ | $\theta$ quanh $X$ | $= X$ khi $\theta = \pi$ (sai khác global phase) |
| $R_Y(\theta)$ | `qc.ry(th, 0)` | $\cos\frac{\theta}{2} I - i\sin\frac{\theta}{2} Y$ | $\theta$ quanh $Y$ | chỉnh xác suất |
| $R_Z(\theta)$ | `qc.rz(th, 0)` | $\mathrm{diag}(e^{-i\theta/2}, e^{i\theta/2})$ | $\theta$ quanh $Z$ | chỉ đổi phase |

Nghịch đảo: `sdg`, `tdg`, hoặc `qc.inverse()` cho cả mạch.

### Thử tự làm

1. Kiểm tra $HXH = Z$ (đối ngẫu với $HZH = X$).
2. Tạo $|i\rangle$ từ $|0\rangle$ chỉ bằng hai cổng (gợi ý: $H$ rồi $S$), đối chiếu với bài 00.
3. Chứng minh bằng số rằng `rz(theta)` không đổi xác suất đo trong cơ sở Z với mọi $\theta$.

Bài tiếp theo: `02_cong_nhieu_qubit.ipynb`.